## Transformaciones: limpieza, resolución de entidades y hechos

Se limpia el staging, se resuelven entidades y proveedores a una clave canónica y se construye la tabla de hechos al grano
**entidad–proveedor–familia**, en `data/final/`

- **Corre en:** Spark. La resolución hace joins y un LSH sobre ~3M menciones; el plan sigue en `a_planteamiento`

### Utilidades

In [1]:
NOTEBOOK = "b_transformaciones"
%run ../funciones.ipynb
%run funciones.ipynb

In [2]:
spark = crear_spark(NOTEBOOK)
verificar_cluster(spark)

Master        : spark://spark-master:7077
Executors     : 2
Cores totales : 2
Memoria driver: 2.0 GB
UI del job    : http://localhost:4040


### Carga del staging

In [3]:
proc_raw   = leer(staging_dir, "procesos")
adj_raw    = leer(staging_dir, "adjudicaciones").drop("anio")
of_raw     = leer(staging_dir, "oferentes").drop("anio")
items_raw  = leer(staging_dir, "items").drop("anio")
con_raw    = leer(staging_dir, "contratos").drop("anio")
partes_raw = leer(staging_dir, "partes").drop("anio")

---
### 1. Limpieza

#### a. Procesos

In [4]:
if not tc_path.exists():
    descargar_tipo_cambio(tc_path, anios)
tipo_cambio = spark.createDataFrame(pd.read_csv(tc_path))

pasos = [("procesos staging", proc_raw.count())]
proc = quitar_duplicados(normalizar_texto(proc_raw, ["entidad", "region", "metodo_detalle", "categoria"]))
pasos.append(("procesos sin ocid duplicado", proc.count()))
proc = proc.where(F.col("anio").isin(anios))
pasos.append((f"procesos con fecha en {min(anios)}-{max(anios)}", proc.count()))

In [5]:
proc = (limpiar_montos(convertir_a_soles(proc, tipo_cambio, "monto_referencial", "moneda", "fecha"), ["monto_referencial"])
        .withColumn("mes", F.month("fecha"))
        .withColumn("postor_unico", (F.col("n_postores") == 1).cast("int"))
        .withColumn("es_directa", F.col("metodo_detalle").rlike("CONTRATACI.N DIRECTA").cast("int"))
        .withColumn("entidad_key", clave_entidad("entidad_id_raw", "entidad"))).cache()
pasos.append(("procesos limpios", proc.count()))

#### b. Adjudicaciones

In [6]:
adj = (adj_raw.join(proc.select("ocid", F.col("fecha").alias("_fp")), "ocid")
              .withColumn("_f", F.coalesce("fecha_adj", "_fp"))
              .withColumn("award_id", F.coalesce("award_id", F.lit("SIN_ID"))))
adj = limpiar_montos(convertir_a_soles(adj, tipo_cambio, "monto_adj", "moneda_adj", "_f"), ["monto_adj"]).drop("_f", "_fp").cache()
pasos += [("adjudicaciones staging (award × proveedor)", adj_raw.count()), ("adjudicaciones en procesos limpios", adj.count())]
pd.DataFrame(pasos, columns=["paso", "filas"])

,paso,filas
0,procesos staging,236229
1,procesos sin ocid duplicado,236229
2,procesos con fecha en 2023-2025,235954
3,procesos limpios,235954
4,adjudicaciones staging (award × proveedor),203486
5,adjudicaciones en procesos limpios,203293


- `convertir_a_soles` usa el tipo de cambio del **mes** del monto; `monto ≤ 0` pasa a nulo
- No se filtra por `estado_adj`: la fuente no publica `awards.status` (100% nulo, `00_EDA/a_entendimiento` §3.e)
- `es_directa` y `postor_unico` quedan como banderas de competencia (las features del P1)

---
### 2. Resolución de entidades

Tres niveles, del más seguro al menos seguro:

1. **Identificador** (determinista): `ENT:`, `RUC:`, `PN:`, `CONS:` o `EXT:` (tabla en `funciones`)
2. **Nombre normalizado** (determinista): una mención sin id con el mismo nombre que **una sola** clave se enlaza a ella
3. **MinHash + LSH** (probabilístico): menciones sin id se enlazan a la clave más parecida si Jaccard ≥ 0.85; si no, nodo
   `NOM:`. Los consorcios solo se enlazan por nombre exacto: sus nombres son genéricos y LSH los confunde

Claves distintas **nunca** se fusionan: dos RUC son dos personas jurídicas. Los casi homónimos se guardan como "posibles
vinculadas" (señal para revisión)

#### a. Entidades compradoras

In [7]:
w = Window.partitionBy("entidad_key").orderBy(F.desc("n"), "entidad")
dim_entidades = (proc.groupBy("entidad_key", "entidad", "entidad_ruc", "region").agg(F.count("*").alias("n"))
                     .withColumn("rk", F.row_number().over(w)).where("rk = 1")
                     .select("entidad_key", F.col("entidad").alias("nombre"), F.col("entidad_ruc").alias("ruc"), "region"))
homonimos = proc.groupBy("entidad").agg(F.countDistinct("entidad_key").alias("claves")).where("claves > 1")
print(f"entidades: {dim_entidades.count():,} claves ({dim_entidades.where(F.col('entidad_key').startswith('ENT:')).count():,} ENT:) | "
      f"{proc.select('entidad').distinct().count():,} nombres | nombres compartidos por varias claves: {homonimos.count():,}")

entidades: 2,996 claves (2,996 ENT:) | 2,995 nombres | nombres compartidos por varias claves: 2


- **2,996 entidades, todas con clave `ENT:`** (código CONSUCODE)
- 2 nombres corresponden a más de una clave: homónimos legítimos (entidades distintas con el mismo nombre), que **no** se
  fusionan

#### b. Catálogo de menciones de proveedores (ganadores + oferentes)

In [8]:
menciones = (adj.select(F.lit("proveedor").alias("rol"), "ocid", F.col("proveedor_id_raw").alias("id_raw"),
                        F.col("proveedor").alias("nombre_raw"))
    .unionByName(of_raw.join(proc.select("ocid"), "ocid")
                 .select(F.lit("oferente").alias("rol"), "ocid", F.col("oferente_id_raw").alias("id_raw"),
                         F.col("oferente").alias("nombre_raw")))
    .withColumn("clave", clave_id("id_raw"))
    .withColumn("nombre_norm", normalizar_nombre("nombre_raw"))
    .where("clave IS NOT NULL OR nombre_norm <> ''")).cache()

w = Window.partitionBy("clave").orderBy(F.desc("n"), "nombre")
canon = (menciones.where("clave IS NOT NULL")
         .groupBy("clave", F.upper(F.trim("nombre_raw")).alias("nombre"), "nombre_norm").agg(F.count("*").alias("n"))
         .withColumn("rk", F.row_number().over(w)).cache())
nombres_clave = canon.where("rk = 1").select("clave", "nombre")
variantes_clave = canon.where("nombre_norm <> ''").select("clave", "nombre_norm").distinct()
sin_clave = menciones.where("clave IS NULL").groupBy("nombre_norm").agg(F.count("*").alias("menciones")).cache()
menciones.groupBy(F.coalesce(F.split("clave", ":")[0], F.lit("sin id")).alias("tipo")).count().orderBy(F.desc("count")).show()

+------+-------+
|  tipo|  count|
+------+-------+
|   RUC|2442463|
|    PN| 465636|
|  CONS| 187746|
|   EXT|   9749|
|sin id|      3|
+------+-------+



- De ~3.1M menciones de proveedores (ganadores + oferentes), **solo 3 no traen identificador**
- 79% `RUC:`, 15% `PN:` (personas naturales), 6% `CONS:` (consorcios), 0.3% `EXT:` (extranjeros)

#### c. Enlace exacto y MinHash/LSH

In [9]:
UMBRAL_LINK, UMBRAL_VINC, MAX_BUCKET = 0.85, 0.95, 100
t0 = time.time()

# 1) nombre sin id idéntico al de una sola clave → enlace directo
exactos = (sin_clave.select("nombre_norm").join(variantes_clave, "nombre_norm")
           .groupBy("nombre_norm").agg(F.min("clave").alias("clave"), F.countDistinct("clave").alias("_n"))
           .where("_n = 1").drop("_n").withColumn("nombre_b", F.col("nombre_norm")).withColumn("jaccard", F.lit(1.0)))

# 2) LSH solo contra claves que no son consorcio
nombres_con = variantes_clave.where(~F.col("clave").startswith("CONS:")).select("nombre_norm").distinct()
pendientes = sin_clave.select("nombre_norm").join(variantes_clave.select("nombre_norm"), "nombre_norm", "left_anti")
sh = materializar(con_shingles(pendientes.unionByName(nombres_con).distinct()), "sh")
bandas = materializar(bandas_lsh(sh), "bandas")

In [10]:
b_con = bandas.join(nombres_con, "nombre_norm")
calientes = b_con.groupBy("banda", "bucket").count().where(f"count > {MAX_BUCKET}")
b_con = materializar(b_con.join(calientes.select("banda", "bucket"), ["banda", "bucket"], "left_anti"), "b_con")
b_pen = bandas.join(pendientes, "nombre_norm")
print(f"pendientes: {pendientes.count():,} | nombres con id (sin consorcios): {nombres_con.count():,} | "
      f"buckets calientes descartados: {calientes.count():,}")

def con_jaccard(pares):
    sa = sh.select(F.col("nombre_norm").alias("nombre_a"), F.col("sh").alias("sh_a"))
    sb = sh.select(F.col("nombre_norm").alias("nombre_b"), F.col("sh").alias("sh_b"))
    return pares.join(sa, "nombre_a").join(sb, "nombre_b").withColumn("jaccard", jaccard("sh_a", "sh_b")).drop("sh_a", "sh_b")

pendientes: 2 | nombres con id (sin consorcios): 114,729 | buckets calientes descartados: 84


In [11]:
# 3) candidatos por bucket compartido → Jaccard exacto → mejor clave por nombre
pares = (b_pen.alias("a").join(b_con.alias("b"), ["banda", "bucket"])
         .select(F.col("a.nombre_norm").alias("nombre_a"), F.col("b.nombre_norm").alias("nombre_b")).distinct())
cand = (con_jaccard(pares).where(f"jaccard >= {UMBRAL_LINK}")
        .join(variantes_clave.withColumnRenamed("nombre_norm", "nombre_b"), "nombre_b"))
w = Window.partitionBy("nombre_a").orderBy(F.desc("jaccard"), "clave")
enlaces = materializar(cand.withColumn("rk", F.row_number().over(w)).where("rk = 1")
                           .select(F.col("nombre_a").alias("nombre_norm"), "clave", "nombre_b", "jaccard")
                           .unionByName(exactos), "enlaces")

mapa_sin = materializar(
    sin_clave.join(enlaces, "nombre_norm", "left")
        .withColumn("metodo", F.when(F.col("clave").isNull(), "nuevo").when(F.col("jaccard") == 1.0, "exacto").otherwise("lsh"))
        .withColumn("clave", F.coalesce("clave", F.concat(F.lit("NOM:"), F.substring(F.sha1("nombre_norm"), 1, 12)))),
    "mapa_sin")
print(f"LSH en {time.time() - t0:,.0f} s")
mapa_sin.groupBy("metodo").count().show()
enlaces.where("jaccard < 1").orderBy("jaccard").show(8, truncate=40)

LSH en 24 s
+------+-----+
|metodo|count|
+------+-----+
| nuevo|    2|
+------+-----+

+-----------+-----+--------+-------+
|nombre_norm|clave|nombre_b|jaccard|
+-----------+-----+--------+-------+
+-----------+-----+--------+-------+



- Con las claves `CONS:` y `EXT:`, **prácticamente todas las menciones se resuelven por identificador**; LSH no encuentra
  enlaces nuevos
- Su aporte principal es la **auditoría** de §2.d: detectar claves distintas con nombre casi idéntico
- En el trabajo previo, sin esas claves, LSH enlazaba consorcios distintos con nombres parecidos ("CONSORCIO ALEJANDRIA" →
  "CONSORCIO ALEJANDRINO"): el `assert` de §6 lo descarta

#### d. Posibles vinculadas (auditoría, no se fusionan)

In [12]:
pares_v = (b_con.alias("x").join(b_con.alias("y"), ["banda", "bucket"])
           .where(F.col("x.nombre_norm") < F.col("y.nombre_norm"))
           .select(F.col("x.nombre_norm").alias("nombre_a"), F.col("y.nombre_norm").alias("nombre_b")).distinct())
vc = variantes_clave
vinculadas = materializar(
    con_jaccard(pares_v).where(f"jaccard >= {UMBRAL_VINC}")
        .join(vc.select(F.col("nombre_norm").alias("nombre_a"), F.col("clave").alias("clave_a")), "nombre_a")
        .join(vc.select(F.col("nombre_norm").alias("nombre_b"), F.col("clave").alias("clave_b")), "nombre_b")
        .where("clave_a <> clave_b"), "vinculadas")
vinculadas.select("nombre_a", "clave_a", "clave_b", "jaccard").show(5, truncate=40)

+----------------------------------+---------------+---------------+------------------+
|                          nombre_a|        clave_a|        clave_b|           jaccard|
+----------------------------------+---------------+---------------+------------------+
|BRAFER INGENIERIA Y CONSTRUCCIONES|RUC:20559869245|RUC:20608801431|0.9696969696969697|
|           SERVICIOS GENERALES GRA|RUC:20495999352|RUC:20531314337|0.9523809523809523|
| HERMES TECHNICAL INTERNACIONAL IN|EXT:L0000002204|EXT:L0000002892|0.9655172413793104|
|   ACR CONSTRUCTORES Y CONSULTORES|RUC:20600198522|RUC:20607876577|0.9565217391304348|
|     A S CONSTRUCTORA Y CONSULTORA|RUC:20606003081|RUC:20608612638|               1.0|
+----------------------------------+---------------+---------------+------------------+
only showing top 5 rows



#### e. Resolver tablas y dimensión de proveedores

In [13]:
mapa_b = F.broadcast(mapa_sin.select("nombre_norm", F.col("clave").alias("_clave_lsh")))

def resolver(df, col_id, col_nombre):
    return (df.withColumn("_clave", clave_id(col_id)).withColumn("nombre_norm", normalizar_nombre(col_nombre))
              .join(mapa_b, "nombre_norm", "left")
              .withColumn("proveedor_key", F.coalesce("_clave", F.when(F.col("nombre_norm") != "", F.col("_clave_lsh"))))
              .drop("_clave", "_clave_lsh", "nombre_norm"))

reg_prov = materializar(
    resolver(partes_raw.where(F.array_contains("roles", "supplier") | F.array_contains("roles", "tenderer"))
                       .select("parte_id_raw", "nombre", "region"), "parte_id_raw", "nombre")
        .where("region IS NOT NULL AND proveedor_key IS NOT NULL")
        .groupBy("proveedor_key", F.upper(F.trim("region")).alias("region")).count()
        .withColumn("rk", F.row_number().over(Window.partitionBy("proveedor_key").orderBy(F.desc("count"), "region")))
        .where("rk = 1").select("proveedor_key", "region"), "reg_prov")

In [14]:
dim_proveedores = materializar(
    nombres_clave.select(F.col("clave").alias("proveedor_key"), "nombre")
        .unionByName(mapa_sin.where("metodo = 'nuevo'").select(F.col("clave").alias("proveedor_key"), F.col("nombre_norm").alias("nombre")))
        .dropDuplicates(["proveedor_key"])
        .withColumn("tipo", F.split("proveedor_key", ":")[0])
        .withColumn("es_consorcio", ((F.col("tipo") == "CONS") | F.col("nombre").rlike("^CONSORCIO")).cast("int"))
        .join(reg_prov, "proveedor_key", "left"), "dim_proveedores")

impacto = pd.DataFrame([
    ("ids crudos distintos", menciones.select("id_raw").distinct().count()),
    ("nombres crudos distintos", menciones.select(F.upper(F.trim("nombre_raw"))).distinct().count()),
    ("claves por identificador", menciones.where("clave IS NOT NULL").select("clave").distinct().count()),
    ("claves con > 1 variante de nombre", canon.where("nombre_norm <> ''").groupBy("clave").agg(F.countDistinct("nombre_norm").alias("v")).where("v > 1").count()),
    ("nombres sin identificador", sin_clave.count()),
    ("  enlazados exactos", mapa_sin.where("metodo = 'exacto'").count()),
    ("  enlazados por LSH", mapa_sin.where("metodo = 'lsh'").count()),
    ("  nodos nuevos NOM:", mapa_sin.where("metodo = 'nuevo'").count()),
    ("pares posibles vinculadas", vinculadas.count()),
    ("nodos proveedor finales", dim_proveedores.count())], columns=["medida", "valor"])
impacto

,medida,valor
0,ids crudos distintos,260077
1,nombres crudos distintos,163168
2,claves por identificador,260074
3,claves con > 1 variante de nombre,544
4,nombres sin identificador,2
5,enlazados exactos,0
6,enlazados por LSH,0
7,nodos nuevos NOM:,2
8,pares posibles vinculadas,71
9,nodos proveedor finales,260076


- **544 claves con más de una variante de nombre:** sin resolución por identificador, cada variante sería un nodo distinto
- **Ids crudos ≈ claves** (260,077 vs. 260,074): en esta fuente cada actor trae un solo formato de id, así que la regla
  RUC 10 / DNI → `PN:` es preventiva
- **71 pares de posibles vinculadas:** claves distintas con nombre casi idéntico (Jaccard ≥ 0.95), por ejemplo empresas
  homónimas o extranjeros registrados dos veces. Se reportan y **no** se fusionan
- **Ética y privacidad:** las claves `PN:` contienen el DNI de personas naturales; en la web solo se publican agregados o
  hashes

---
### 3. Hechos (grano entidad–proveedor–familia)

#### a. Hechos

In [15]:
w_aw = Window.partitionBy("ocid", "award_id")
adj_r = (resolver(adj, "proveedor_id_raw", "proveedor").where("proveedor_key IS NOT NULL")
         .dropDuplicates(["ocid", "award_id", "proveedor_key"])
         .withColumn("n_proveedores_adj", F.count("*").over(w_aw)))

con_link = con_raw.select("ocid", "award_id").distinct().withColumn("tiene_contrato", F.lit(1))
fam_tender = (items_raw.where("unspsc IS NOT NULL")
              .groupBy("ocid").agg(F.array_distinct(F.collect_list(F.substring("unspsc", 1, 4))).alias("fam_tender")))
fam_award = F.array_distinct(F.transform(F.filter("unspsc_adj", lambda x: x.isNotNull()), lambda x: F.substring(x, 1, 4)))

In [16]:
hechos = (adj_r
    .join(proc.select("ocid", "fecha", "anio", "mes", "entidad_key", "region", "categoria", "metodo_detalle",
                      "postor_unico", "es_directa", "n_postores"), "ocid")
    .join(fam_tender, "ocid", "left")
    .join(con_link, ["ocid", "award_id"], "left").fillna({"tiene_contrato": 0})
    .withColumn("_fa", fam_award)
    .withColumn("familias", F.coalesce(F.when(F.size("_fa") > 0, F.col("_fa")),
                                       F.when(F.size("fam_tender") > 0, F.col("fam_tender")), F.array(F.lit("SIN"))))
    .withColumn("monto_asignado", F.col("monto_adj") / F.col("n_proveedores_adj") / F.size("familias"))
    .select("ocid", "award_id", "fecha", "anio", "mes", "entidad_key", "proveedor_key", "region", "categoria",
            "metodo_detalle", "postor_unico", "es_directa", "n_postores", "n_proveedores_adj", "tiene_contrato",
            "monto_adj", "monto_asignado", F.explode("familias").alias("familia"))
    .withColumn("segmento", F.substring("familia", 1, 2)))

- **Arista por adjudicación**, no por contrato: 10.8% de adjudicaciones no tiene contrato vinculado
  (`a_entendimiento` §3.d). El contrato queda como `tiene_contrato`; los contratos sin adjudicación (1.0%) no
  generan arista porque no tienen proveedor
- **Familia UNSPSC (4 dígitos) como rubro:** primero los ítems del award; si no, los del tender; si no, `SIN`
- **`monto_asignado` fraccional** (monto / ganadores / familias): Σ monto_asignado = monto adjudicado, sin contar dos veces

#### b. Participantes (oferentes ∪ ganadores)

In [17]:
of_r = (resolver(of_raw.join(proc.select("ocid"), "ocid"), "oferente_id_raw", "oferente")
        .where("proveedor_key IS NOT NULL").select("ocid", "proveedor_key").distinct()
        .withColumn("en_lista_oferentes", F.lit(1)))
gan = adj_r.select("ocid", "proveedor_key").distinct().withColumn("gano", F.lit(1))
participantes = (of_r.join(gan, ["ocid", "proveedor_key"], "full").fillna(0, ["en_lista_oferentes", "gano"])
                 .join(proc.select("ocid", "anio", "mes"), "ocid"))

- Un ganador que no aparece en `tenderers` (p. ej. contratación directa) se agrega igual: participó y ganó
- Un mismo proveedor resuelto aparece una sola vez por proceso (`distinct`), para no inflar la co-participación

---
### 4. Escritura en `data/final/`

In [18]:
escribir = {"procesos": (proc, ["anio", "mes"]), "hechos": (hechos, ["anio", "mes"]),
            "participantes": (participantes, ["anio", "mes"]),
            "entidades": (dim_entidades.coalesce(1), []), "proveedores": (dim_proveedores.coalesce(1), []),
            "er_mapa_lsh": (mapa_sin.coalesce(1), []), "er_posibles_vinculadas": (vinculadas.coalesce(1), [])}
for nombre, (df, parts) in escribir.items():
    w_ = df.write.mode("overwrite")
    (w_.partitionBy(*parts) if parts else w_).parquet(str(final_dir / nombre))
print("data/final:", ", ".join(escribir))

data/final: procesos, hechos, participantes, entidades, proveedores, er_mapa_lsh, er_posibles_vinculadas


- `procesos`, `hechos` y `participantes` se particionan por `anio/mes`: el hold-out temporal (2025) y los análisis por
  periodo leen solo esas carpetas (*partition pruning*, ver `c_uso` §4)
- Las dimensiones y la auditoría de la resolución son chicas: un solo archivo

---
### 5. Reconciliación

Filas en cada paso, leídas de lo **ya escrito**, para detectar pérdidas no explicadas

In [19]:
fin = {n: leer(final_dir, n) for n in ("procesos", "hechos", "participantes")}
reconciliacion = pd.DataFrame(pasos + [
    ("adjudicaciones con proveedor resuelto (award × proveedor)", adj_r.count()),
    ("hechos: award × proveedor distintos", fin["hechos"].select("ocid", "award_id", "proveedor_key").distinct().count()),
    ("hechos: filas (× familia)", fin["hechos"].count()),
    ("oferentes staging", of_raw.count()),
    ("oferentes en procesos limpios", of_raw.join(proc.select("ocid"), "ocid").count()),
    ("participantes (proceso × proveedor resuelto)", fin["participantes"].count())], columns=["paso", "filas"])
reconciliacion

,paso,filas
0,procesos staging,236229
1,procesos sin ocid duplicado,236229
2,procesos con fecha en 2023-2025,235954
3,procesos limpios,235954
4,adjudicaciones staging (award × proveedor),203486
5,adjudicaciones en procesos limpios,203293
6,adjudicaciones con proveedor resuelto (award × proveedor),199377
7,hechos: award × proveedor distintos,199377
8,hechos: filas (× familia),202966
9,oferentes staging,2906469


- **Procesos:** 236,229 → 235,954 (se quitan los 275 fuera de 2023-2025)
- **Adjudicaciones:** 203,486 → 203,293 en procesos limpios → **199,377 con proveedor resuelto**. La diferencia (3,916) son
  los awards sin proveedor (§3.a del EDA)
- **Hechos:** 199,377 award × proveedor → 202,966 filas al explotar por familia (+3,589 por awards con varias familias)
- **Participantes:** 2,906,220 oferentes + 22 ganadores no listados = 2,906,242
- **No hay pérdidas no explicadas.** En el trabajo previo, un `cache()` tras un join perdía el 63% de las filas por un bug
  de Spark 3.5.0 (SPARK-45592), corregido en `crear_spark`

---
### 6. Escritura de resultados

In [20]:
escritos = guardar_tablas({"reconciliacion": reconciliacion, "impacto_resolucion": impacto,
                           "enlaces_lsh_muestra": enlaces.where("jaccard < 1").orderBy("jaccard").limit(50),
                           "vinculadas_muestra": vinculadas.limit(50)})

Escrito en 01_Trans/temp/b_transformaciones/: reconciliacion, impacto_resolucion, enlaces_lsh_muestra, vinculadas_muestra


#### a. Verificación

In [21]:
verificar_salidas(escritos)
h = fin["hechos"]
assert h.where(~F.col("entidad_key").startswith("ENT:")).count() == 0, "hay entidades sin código CONSUCODE"
cons_mal = leer(final_dir, "er_mapa_lsh").where("metodo = 'lsh' AND clave LIKE 'CONS:%'").count()
assert cons_mal == 0, "LSH enlazó nombres a consorcios"
print("claves de entidad: todas ENT: | enlaces LSH a consorcios: 0")

OK: 4 tablas (parquet + json)


claves de entidad: todas ENT: | enlaces LSH a consorcios: 0


---
### 7. Cerrar sesión

In [22]:
spark.stop()